# Extract genes based on PCA

## PCA

In [1]:
import pandas as pd
%run "../Model/DataHelpers.ipynb"

In [2]:
print(f'Loading gene data - Start')
df = pd.read_csv('../Data/geneDataPreProcessed.csv')
print(f'Loading gene data - End')

Loading gene data - Start
Loading gene data - End


### Dataset split: training and test data

In [3]:
X, y, X_train, X_test, y_train, y_test, test_case_ids = split_data(df, "tnbc", True)

X_train.shape=(781, 19938)
X_test.shape=(196, 19938)
y_train.shape=(781,)
y_test.shape=(196,)


# Apply PCA

In [4]:
variant = FeatureVariant.AUTOMATED
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Scale features (fit ONLY on train)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Apply PCA (fit ONLY on train)
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"Original features: {X_train.shape[1]}")
print(f"PCA components retained: {X_train_pca.shape[1]}")

# Build PCA DataFrames
pca_columns = [f'PC{i+1}' for i in range(X_train_pca.shape[1])]

df_train_pca = pd.DataFrame(X_train_pca, columns=pca_columns)
df_train_pca['tnbc'] = y_train.values

df_test_pca = pd.DataFrame(X_test_pca, columns=pca_columns)
df_test_pca['tnbc'] = y_test.values
df_test_pca['case_id'] = test_case_ids.values


Original features: 19938
PCA components retained: 535


# Send dataframe to a csv

In [5]:
# Write PCA features to disk
# Separate train and test for use in models
df_train_pca.to_csv(f'../Data/interim/patient_genes_{variant}_train.csv', index=False)
df_test_pca.to_csv(f'../Data/interim/patient_genes_{variant}_test.csv', index=False)